In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import os
from torchvision import transforms
transform = transforms.Compose([
transforms.Resize((32, 32)), # Resize images
transforms.RandomRotation(15), # Rotate images randomly within ±15 de
transforms.ToTensor(), # Convert to tensor
transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]), # value
])

transform_2 = transforms.Compose([
transforms.Resize((32, 32)), # Resize images
transforms.ToTensor(), # Convert to tensor
transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]), # value
])

train_path = os.path.join(path, '/PlantVillage/train')
test_path = os.path.join(path, '/PlantVillage/test')
train_dataset = ImageFolder('/kaggle/input/q1-stage-3-2026/PlantVillage/train', transform=transform)
test_dataset = ImageFolder('/kaggle/input/q1-stage-3-2026/PlantVillage/test', transform=transform_2)





In [ ]:
# Write your code here
import torch.nn as nn
import torch
class CNNModel(nn.Module):
  def __init__(self, num_classes):
    super(CNNModel, self).__init__()
    self.features = nn.Sequential(
      nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3),
      nn.ReLU(),
      nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size=2, stride=2),
      nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
      nn.ReLU(),
      nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3),
      nn.ReLU(),
      nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3),
      nn.ReLU(),
)
    self.classifier = nn.Sequential(
      nn.Linear(64, 256),
      nn.Linear( 256,256)
      )
  def forward(self, x):
    x = self.features(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x

In [ ]:
# Write your code here
from tqdm import tqdm
def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  print(device)
  total_loss = 0
  correct = 0
  total = 0
  for images, labels in tqdm(dataloader):
    images, labels = images.to(device), labels
    outputs = model(images)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    outputs = torch.softmax(outputs, dim=1)
    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)
  avg_loss = total_loss / len(dataloader)
  accuracy = 100 * correct / total
  return avg_loss, accuracy

In [ ]:
from tqdm import tqdm
def  validate(model, dataloader, criterion, device):
  model.eval()
  print(device)
  total_loss = 0
  correct = 0
  total = 0
  with torch.no_grad():
    for images, labels in tqdm(dataloader):
      images, labels = images.to(device), labels
      outputs = model(images)
      loss = criterion(outputs, labels)
      total_loss += loss.item()
      outputs = torch.softmax(outputs, dim=1)
      predictions = outputs.argmax(dim=1)
      correct += (predictions == labels).sum().item()
      total += labels.size(0)
  avg_loss = total_loss / len(dataloader)
  accuracy = 100 * correct / total
  return avg_loss, accuracy

In [ ]:
# Write your code here
import torch.optim as optim
dataloader= train_dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = CNNModel(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
for epoch in range(num_epochs):
  train_loss, train_accuracy = train_one_epoch(model, dataloader, criterion, optimizer, device)
  val_loss, val_accuracy = validate(model, dataloader, criterion, device)
  train_losses.append(train_loss)
  val_losses.append(val_loss)
  train_accuracies.append(train_accuracy)
  val_accuracies.append(val_accuracy)
  print(f"Epoch {epoch+1}/{num_epochs}: "
  f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2}"
  f"Val Loss={val_loss:.4f}, Val_Accuracy={val_accuracy:.2f}%")


In [ ]:
# Write your code here
